In [1]:
import pandas as pd
import numpy as np
import ast
from datetime import datetime

In [2]:
df = pd.read_csv('games_march2025_cleaned.csv')

In [3]:
df.head()

,appid,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,...,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags,pct_pos_total,num_reviews_total,pct_pos_recent,num_reviews_recent
0,730,Counter-Strike 2,2012-08-21,0,0.00,1,"For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...",NaN,...,879,5174,350,0,1212356,"{'FPS': 90857, 'Shooter': 65397, 'Multiplayer'...",86,8632939,82,96473
1,578080,PUBG: BATTLEGROUNDS,2017-12-21,0,0.00,0,"LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...","LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...",Play PUBG: BATTLEGROUNDS for free. Land on str...,NaN,...,0,0,0,0,616738,"{'Survival': 14838, 'Shooter': 12727, 'Battle ...",59,2513842,68,16720
2,570,Dota 2,2013-07-09,0,0.00,2,"The most-played game on Steam. Every day, mill...","The most-played game on Steam. Every day, mill...","Every day, millions of players worldwide enter...",“A modern multiplayer masterpiece.” 9.5/10 – D...,...,1536,898,892,0,555977,"{'Free to Play': 59933, 'MOBA': 20158, 'Multip...",81,2452595,80,29366
3,271590,Grand Theft Auto V Legacy,2015-04-13,17,0.00,0,"When a young street hustler, a retired bank ro...","When a young street hustler, a retired bank ro...",Grand Theft Auto V for PC offers players the o...,NaN,...,771,7101,74,0,117698,"{'Open World': 32644, 'Action': 23539, 'Multip...",87,1803832,92,17517
4,359550,Tom Clancy's Rainbow Six® Siege,2015-12-01,17,3.99,9,Edition Comparison Ultimate Edition The Tom Cl...,“One of the best first-person shooters ever ma...,"Tom Clancy's Rainbow Six® Siege is an elite, t...",NaN,...,682,2434,306,80,89916,"{'FPS': 9831, 'PvP': 9162, 'e-sports': 9072, '...",84,1168020,76,12608


In [13]:
cleaned_df = df.copy()

In [14]:
df.columns


Index(['appid', 'name', 'release_date', 'required_age', 'price', 'dlc_count',
       'detailed_description', 'about_the_game', 'short_description',
       'reviews', 'header_image', 'website', 'support_url', 'support_email',
       'windows', 'mac', 'linux', 'metacritic_score', 'metacritic_url',
       'achievements', 'recommendations', 'notes', 'supported_languages',
       'full_audio_languages', 'packages', 'developers', 'publishers',
       'categories', 'genres', 'screenshots', 'movies', 'user_score',
       'score_rank', 'positive', 'negative', 'estimated_owners',
       'average_playtime_forever', 'average_playtime_2weeks',
       'median_playtime_forever', 'median_playtime_2weeks', 'discount',
       'peak_ccu', 'tags', 'pct_pos_total', 'num_reviews_total',
       'pct_pos_recent', 'num_reviews_recent'],
      dtype='object')

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89618 entries, 0 to 89617
Data columns (total 47 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   appid                     89618 non-null  int64  
 1   name                      89618 non-null  object 
 2   release_date              89618 non-null  object 
 3   required_age              89618 non-null  int64  
 4   price                     89618 non-null  float64
 5   dlc_count                 89618 non-null  int64  
 6   detailed_description      89421 non-null  object 
 7   about_the_game            89398 non-null  object 
 8   short_description         89498 non-null  object 
 9   reviews                   10401 non-null  object 
 10  header_image              89618 non-null  object 
 11  website                   41114 non-null  object 
 12  support_url               44110 non-null  object 
 13  support_email             78798 non-null  object 
 14  window

In [16]:
numeric_cols = ['appid','required_age', 'metacritic_score', 
                'achievements', 'recommendations','user_score', 'positive', 'negative'
                'average_playtime_forever', 'average_playtime_2weeks', 'median_playtime_forever',
                'median_playtime_2weeks', 'discount', 'peak_ccu', 'pct_pos_total', 'num_reviews_total', 
                'pct_pos_recent', 'num_reviews_recent']

In [17]:
 for col in numeric_cols:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].fillna(0)

In [18]:
  # Text columns
    text_cols = ['name', 'release_date','detailed_description', 'about_the_game', 'short_description',
                'reviews','headerimage', 'website', 'notes', 'supported_languages', 'full_audio_languages']

IndentationError: unexpected indent (2728523110.py, line 2)

In [19]:
cleaned_df['release_date'] = pd.to_datetime(cleaned_df['release_date'], errors='coerce')
    
    # Handle missing values
cleaned_df['price'] = cleaned_df['price'].fillna(0)
cleaned_df['metacritic_score'] = cleaned_df['metacritic_score'].fillna(0)
cleaned_df['dlc_count'] = cleaned_df['dlc_count'].fillna(0)
    
    # Fix encoding issues in text fields
text_columns = ['name', 'detailed_description', 'about_the_game', 'short_description']
for col in text_columns:
    if col in cleaned_df.columns:
        cleaned_df[col] = cleaned_df[col].apply(lambda x: x.encode('ascii', 'ignore').decode('ascii') if isinstance(x, str) else x)
    
    # Convert estimated_owners to numeric
    if 'estimated_owners' in cleaned_df.columns:
        # Extract mean value from range (e.g., "100000 - 200000" -> 150000)
        def extract_owner_mean(owner_range):
            if pd.isna(owner_range) or not isinstance(owner_range, str):
                return np.nan
            try:
                parts = owner_range.split(' - ')
                if len(parts) == 2:
                    # Remove commas and convert to integers
                    lower = int(parts[0].replace(',', ''))
                    upper = int(parts[1].replace(',', ''))
                    return (lower + upper) / 2
                else:
                    return np.nan
            except:
                return np.nan
        
        cleaned_df['estimated_owners_mean'] = cleaned_df['estimated_owners'].apply(extract_owner_mean)
    
    # Calculate review ratio
    if 'positive' in cleaned_df.columns and 'negative' in cleaned_df.columns:
        total_reviews = cleaned_df['positive'] + cleaned_df['negative']
        cleaned_df['review_ratio'] = cleaned_df['positive'] / total_reviews.replace(0, 1)  # Avoid division by zero

    # Clean up boolean columns
    for col in ['windows', 'mac', 'linux']:
        if col in cleaned_df.columns:
            cleaned_df[col] = cleaned_df[col].map({'TRUE': 1, 'FALSE': 0, True: 1, False: 0})
    
print(f"Data cleaning completed. Shape: {cleaned_df.shape}")
    

Data cleaning completed. Shape: (89618, 49)


In [20]:
cleaned_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89618 entries, 0 to 89617
Data columns (total 49 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   appid                     89618 non-null  int64         
 1   name                      89618 non-null  object        
 2   release_date              89618 non-null  datetime64[ns]
 3   required_age              89618 non-null  int64         
 4   price                     89618 non-null  float64       
 5   dlc_count                 89618 non-null  int64         
 6   detailed_description      89421 non-null  object        
 7   about_the_game            89398 non-null  object        
 8   short_description         89498 non-null  object        
 9   reviews                   10401 non-null  object        
 10  header_image              89618 non-null  object        
 11  website                   41114 non-null  object        
 12  support_url       

In [21]:
drop_columns = [
    # Identifiers and metadata
    'appid', 'name', 'header_image',
    
    # URLs and external links
    'website', 'support_url', 'support_email', 'metacritic_url',
    
    # Raw text
    'detailed_description', 'about_the_game', 'short_description',
    'reviews', 'notes',
    
    # Media assets
    'screenshots', 'movies',
    
    # High-cardinality categoricals
    'developers', 'publishers',
    
    # Language lists
    'supported_languages', 'full_audio_languages',
    
    # Redundant or intermediate columns
    'estimated_owners', 'user_score', 'score_rank',
    'packages', 'release_date'  # (keep the engineered date features instead)
]

In [22]:
final_df = df.drop(columns=drop_columns)

In [24]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89618 entries, 0 to 89617
Data columns (total 24 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   required_age              89618 non-null  int64  
 1   price                     89618 non-null  float64
 2   dlc_count                 89618 non-null  int64  
 3   windows                   89618 non-null  bool   
 4   mac                       89618 non-null  bool   
 5   linux                     89618 non-null  bool   
 6   metacritic_score          89618 non-null  int64  
 7   achievements              89618 non-null  int64  
 8   recommendations           89618 non-null  int64  
 9   categories                89618 non-null  object 
 10  genres                    89618 non-null  object 
 11  positive                  89618 non-null  int64  
 12  negative                  89618 non-null  int64  
 13  average_playtime_forever  89618 non-null  int64  
 14  averag

In [39]:
numeric_cols = ['required_age', 'dlc_count', 'metacritic_score', 
                'achievements', 'recommendations', 'positive', 'negative'
                'average_playtime_forever', 'average_playtime_2weeks', 'median_playtime_forever',
                'median_playtime_2weeks', 'discount', 'peak_ccu', 'pct_pos_total', 'num_reviews_total', 
                'pct_pos_recent', 'num_reviews_recent']

In [40]:
for col in numeric_cols:
        if col in final_df.columns:
            final_df[col] = final_df[col].fillna(0)

In [41]:
 # Text columns
text_cols = ['categories', 'genres', 'tags']
    
for col in text_cols:
    if col in final_df.columns:
        final_df[col] = final_df[col].fillna('')

In [42]:
for col in text_cols:
    if col in final_df.columns:
            try:
                final_df[col] = final_df[col].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) and x != '' else [])
            except:
                # Fallback for malformed lists
                final_df[col] = final_df[col].str.strip("[]").str.replace("'", "").str.split(", ")

AttributeError: Can only use .str accessor with string values!

In [ ]:
if 'tags' in final_df.columns:
        try:
            final_df['tags'] = final_df['tags'].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) and x != '' else {})
        except:
            final_df['tags'] = final_df['tags'].apply(lambda x: {})

In [36]:
# Date conversion
if 'release_date' in final_df.columns:
        final_df['release_date'] = pd.to_datetime(final_df['release_date'], errors='coerce')

In [37]:
# Boolean conversion
bool_cols = ['windows', 'mac', 'linux']
for col in bool_cols:
        if col in final_df.columns:
            final_df[col] = final_df[col].map({'TRUE': 1, 'FALSE': 0, True: 1, False: 0})

In [38]:
final_df.head()

,required_age,price,dlc_count,windows,mac,linux,metacritic_score,achievements,recommendations,categories,...,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags,pct_pos_total,num_reviews_total,pct_pos_recent,num_reviews_recent
0,0,0.00,1,1,0,1,0,1,4401572,"[Multi-player, Cross-Platform Multiplayer, Ste...",...,879,5174,350,0,1212356,"{'FPS': 90857, 'Shooter': 65397, 'Multiplayer'...",86,8632939,82,96473
1,0,0.00,0,1,0,0,0,37,1732007,"[Multi-player, PvP, Online PvP, Stats, Remote ...",...,0,0,0,0,616738,"{'Survival': 14838, 'Shooter': 12727, 'Battle ...",59,2513842,68,16720
2,0,0.00,2,1,1,1,90,0,14337,"[Multi-player, Co-op, Steam Trading Cards, Ste...",...,1536,898,892,0,555977,"{'Free to Play': 59933, 'MOBA': 20158, 'Multip...",81,2452595,80,29366
3,17,0.00,0,1,0,0,96,77,1803063,"[Single-player, Multi-player, PvP, Online PvP,...",...,771,7101,74,0,117698,"{'Open World': 32644, 'Action': 23539, 'Multip...",87,1803832,92,17517
4,17,3.99,9,1,0,0,0,0,1165929,"[Single-player, Multi-player, PvP, Online PvP,...",...,682,2434,306,80,89916,"{'FPS': 9831, 'PvP': 9162, 'e-sports': 9072, '...",84,1168020,76,12608
